# Aviation Accidents Analysis

This notebook cleans the aviation accident dataset for the airline/aircraft insurer. It retains professionally built airplanes from 1983 onward, constructs injury and destruction measures, standardizes relevant fields, and saves a cleaned CSV for analysis.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- Inspect NaNs, datatypes, and summary statistics.

In [2]:
paths = [Path("data/AviationData.csv"), Path("AviationData.csv")]
raw_path = next((p for p in paths if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError("Place AviationData.csv in the data folder or repository root.")

df = pd.read_csv(raw_path, encoding="latin1", low_memory=False)
print("File:", raw_path)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing").head(20))
display(df.describe(include="all").T)

File: data\AviationData.csv
Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


,dtype
Event.Id,str
Investigation.Type,str
Accident.Number,str
Event.Date,str
Location,str
Country,str
Latitude,str
Longitude,str
Airport.Code,str
Airport.Name,str


,missing
Schedule,76307
Air.carrier,72241
FAR.Description,56866
Aircraft.Category,56602
Longitude,54516
Latitude,54507
Airport.Code,38757
Airport.Name,36185
Broad.phase.of.flight,27165
Publication.Date,13771


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We retain only airplanes, exclude amateur builds, and keep events dated from 1 January 1983. Missing scope fields are not imputed because that could incorrectly classify an aircraft as eligible.

In [3]:
for col in ["Aircraft.Category", "Amateur.Built", "Event.Date"]:
    display(df[col].value_counts(dropna=False).head(15))

df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df["Aircraft.Category"] = df["Aircraft.Category"].astype("string").str.strip().str.lower()
df["Amateur.Built"] = df["Amateur.Built"].astype("string").str.strip().str.lower()

clean_df = df.loc[
    df["Aircraft.Category"].eq("airplane") &
    df["Amateur.Built"].isin(["no", "n"]) &
    df["Event.Date"].ge("1983-01-01")
].copy()
print("After scope filters:", clean_df.shape)

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
Name: count, dtype: int64

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

Event.Date
1982-05-16    25
1984-06-30    25
2000-07-08    25
1983-06-05    24
1983-08-05    24
1984-08-25    24
1986-05-17    24
1982-05-01    23
1982-07-09    23
1982-10-03    23
1983-05-28    23
1988-08-07    23
2001-06-16    23
2001-07-21    23
1982-04-25    22
Name: count, dtype: int64

After scope filters: (21447, 31)


### Cleaning and constructing Key Measurables

Injury fields are converted to numeric. Where at least one injury field is present, missing injury categories are treated as zero. Rows with no injury information remain missing.

**Construct metric for fatal/serious injuries**

`Persons.Aboard` is estimated as fatal + serious + minor + uninjured. `Fatal.Serious.Injury.Fraction` is (fatal + serious) / persons aboard.

In [4]:
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
for col in injury_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")
has_injury_data = clean_df[injury_cols].notna().any(axis=1)
clean_df.loc[has_injury_data, injury_cols] = clean_df.loc[has_injury_data, injury_cols].fillna(0)
clean_df["Persons.Aboard"] = clean_df[injury_cols].sum(axis=1, min_count=1)
clean_df["Fatal.Serious.Injuries"] = clean_df["Total.Fatal.Injuries"] + clean_df["Total.Serious.Injuries"]
clean_df["Fatal.Serious.Injury.Fraction"] = np.where(
    clean_df["Persons.Aboard"] > 0,
    clean_df["Fatal.Serious.Injuries"] / clean_df["Persons.Aboard"], np.nan)
clean_df.loc[~clean_df["Fatal.Serious.Injury.Fraction"].between(0,1,inclusive="both"), "Fatal.Serious.Injury.Fraction"] = np.nan
display(clean_df[["Persons.Aboard","Fatal.Serious.Injuries","Fatal.Serious.Injury.Fraction"]].describe())

,Persons.Aboard,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction
count,21400.000000,21400.000000,20543.000000
mean,8.611168,0.928598,0.283971
std,35.947238,6.823994,0.431704
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,0.800000
max,588.000000,295.000000,1.000000


**Aircraft.Damage**

Damage labels are standardized. `Aircraft.Destroyed` is 1 for destroyed, 0 for other known damage, and missing where damage is unknown.

In [5]:
# Clean the Aircraft.damage column
clean_df["Aircraft.damage"] = (
    clean_df["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert unknown/missing labels to NaN
clean_df["Aircraft.damage"] = clean_df["Aircraft.damage"].replace({
    "Unknown": pd.NA,
    "": pd.NA
})

# Create destroyed indicator
clean_df["Aircraft.Destroyed"] = clean_df["Aircraft.damage"].map({
    "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0
})

# Check results
display(clean_df["Aircraft.damage"].value_counts(dropna=False))
display(clean_df["Aircraft.Destroyed"].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
<NA>            1324
Minor            817
Name: count, dtype: Int64

Aircraft.Destroyed
0.0    17807
1.0     2316
NaN     1324
Name: count, dtype: int64

### Investigate the *Make* column
- Trim spaces and standardize case.
- Remove blanks and unknown labels.
- Retain makes with at least 50 eligible records, as required by the lab.

In [6]:
clean_df["Make"] = clean_df["Make"].astype("string").str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
clean_df["Make"] = clean_df["Make"].replace({"": pd.NA, "UNKNOWN": pd.NA})
clean_df = clean_df.dropna(subset=["Make"]).copy()
make_counts = clean_df["Make"].value_counts()
eligible_makes = make_counts[make_counts >= 50].index
clean_df = clean_df[clean_df["Make"].isin(eligible_makes)].copy()
display(make_counts.loc[eligible_makes].to_frame("records"))

,records
Make,
CESSNA,7146
PIPER,3989
BEECH,1431
BOEING,1264
MOONEY,363
AIRBUS,243
CIRRUS DESIGN CORP,220
BELLANCA,219
AIR TRACTOR INC,219


### Inspect Model column
- Remove missing and blank model values.
- Standardize labels.
- Combine make and model because a model label may occur under multiple manufacturers.

In [7]:
clean_df["Model"] = clean_df["Model"].astype("string").str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
clean_df["Model"] = clean_df["Model"].replace({"": pd.NA, "UNKNOWN": pd.NA})
clean_df = clean_df.dropna(subset=["Model"]).copy()
model_make_counts = clean_df.groupby("Model")["Make"].nunique().sort_values(ascending=False)
print("Model labels used by multiple makes:", (model_make_counts > 1).sum())
clean_df["Plane.Type"] = clean_df["Make"] + " | " + clean_df["Model"]
display(clean_df[["Make","Model","Plane.Type"]].head())

Model labels used by multiple makes: 118


,Make,Model,Plane.Type
4150,BOEING,747,BOEING | 747
4171,PIPER,PA-28-140,PIPER | PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND | DHC-6
6760,BOEING,727-200,BOEING | 727-200
6806,BEECH,C35,BEECH | C35


### Cleaning other columns
Text fields are trimmed and standardized. Blank, unknown, and UNK labels are treated as missing. Engine count is numeric and values outside 1–8 are treated as invalid.

In [8]:
cat_cols = ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]
for col in cat_cols:
    clean_df[col] = clean_df[col].astype("string").str.strip().str.title().str.replace(r"\s+", " ", regex=True)
    clean_df[col] = clean_df[col].replace({"": pd.NA, "Unknown": pd.NA, "Unk": pd.NA, "Not Applicable": pd.NA})
clean_df["Number.of.Engines"] = pd.to_numeric(clean_df["Number.of.Engines"], errors="coerce")
clean_df.loc[~clean_df["Number.of.Engines"].between(1,8), "Number.of.Engines"] = np.nan
for col in cat_cols + ["Number.of.Engines"]:
    display(clean_df[col].value_counts(dropna=False).head(15))

Engine.Type
Reciprocating      12835
<NA>                3320
Turbo Prop           931
Turbo Fan            701
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
Name: count, dtype: Int64

Weather.Condition
Vmc     14295
<NA>     2679
Imc       905
Name: count, dtype: Int64

Purpose.of.flight
Personal                     9844
<NA>                         3350
Instructional                2410
Aerial Application            724
Business                      409
Positioning                   269
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/Corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Name: count, dtype: Int64

Broad.phase.of.flight
<NA>           15438
Landing         1110
Takeoff          425
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-Around         81
Descent           62
Climb             52
Standing          35
Other              2
Name: count, dtype: Int64

Number.of.Engines
1.0    13222
2.0     2470
NaN     2094
4.0       67
3.0       26
Name: count, dtype: int64

### Column Removal
The lab requires retaining original columns with more than 20,000 non-null observations. Required derived and analysis fields are retained regardless of this threshold.

In [9]:
non_null = clean_df.notna().sum()
base_cols = non_null[non_null > 20000].index.tolist()
required = ["Event.Date","Make","Model","Plane.Type","Aircraft.Damage","Aircraft.Destroyed",
            "Persons.Aboard","Fatal.Serious.Injuries","Fatal.Serious.Injury.Fraction",
            "Engine.Type","Weather.Condition","Number.of.Engines","Purpose.of.flight","Broad.phase.of.flight"]
keep = list(dict.fromkeys(base_cols + [c for c in required if c in clean_df.columns]))
clean_df = clean_df[keep].copy()
print("Cleaned shape:", clean_df.shape)
display(clean_df.head())

Cleaned shape: (17879, 13)


,Event.Date,Make,Model,Plane.Type,Aircraft.Destroyed,Persons.Aboard,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
4150,1983-03-18,BOEING,747,BOEING | 747,0.0,588.0,0.0,0.0,Turbo Fan,Vmc,4.0,<NA>,Taxi
4171,1983-03-20,PIPER,PA-28-140,PIPER | PA-28-140,1.0,2.0,2.0,1.0,Reciprocating,Imc,1.0,Personal,Cruise
4285,1983-04-02,DE HAVILLAND,DHC-6,DE HAVILLAND | DHC-6,NaN,5.0,1.0,0.2,Turbo Prop,Vmc,2.0,Skydiving,Standing
6760,1983-11-08,BOEING,727-200,BOEING | 727-200,0.0,100.0,0.0,0.0,Turbo Fan,Vmc,3.0,<NA>,Taxi
6806,1983-11-13,BEECH,C35,BEECH | C35,0.0,1.0,0.0,0.0,Reciprocating,Vmc,1.0,Personal,Climb


### Save DataFrame to csv
The cleaned data is saved in the repository `data` folder for the analysis notebook.

In [10]:
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "AviationData_Cleaned.csv"
clean_df.to_csv(output_path, index=False)
print(f"Saved {len(clean_df):,} rows to {output_path.resolve()}")

Saved 17,879 rows to C:\Users\ejtmwangi\Documents\summative labs\dsc-course0-m8-lab\data\AviationData_Cleaned.csv
